In [1]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import pandas as pd
import os
from sklearn.metrics import r2_score
from transformers import AutoTokenizer, AutoModelForMaskedLM, AutoModel, get_linear_schedule_with_warmup


/home/christ/Downloads/Telegram Desktop/M1/M1 Avignon universite/approches neuronales/TP/env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
folder=os.listdir('data')
train_file=folder[1]
test_file=folder[0]
val_file=folder[2]


In [3]:
def load_data(file_path):
    with open(os.path.join('data',file_path),'r') as f:
        data=pd.read_csv(f)
    return data

In [4]:
data_train=load_data(train_file)
data_test=load_data(test_file)
data_val=load_data(val_file)

data_train['Review'].head()

0    nice hotel expensive parking got good deal sta...
1    ok nothing special charge diamond member hilto...
2    nice rooms not 4* experience hotel monaco seat...
3    unique, great stay, wonderful time hotel monac...
4    great stay great stay, went seahawk game aweso...
Name: Review, dtype: object

In [5]:
# %%
model_id = "answerdotai/ModernBERT-base"
tokenizer = AutoTokenizer.from_pretrained(model_id)

def tokenize_data(texts, max_length=64):
    return tokenizer(
        texts.tolist(),
        padding=True,
        truncation=True,
        max_length=max_length,  # 64 pour GPU 6 Go (OOM); 128 si plus de VRAM
        return_tensors="pt"
    )

# Prepare labels (Subtract 1 if your ratings are 1-5 to get 0-4 range)
def prepare_labels(df):
    return torch.tensor(df['Rating'].values - 1, dtype=torch.long)

train_labels = prepare_labels(data_train)
val_labels = prepare_labels(data_val)
test_labels = prepare_labels(data_test)

In [6]:
class RegressionModel(nn.Module):
    def __init__(self, model_name="answerdotai/ModernBERT-base", num_classes=5, use_mean_pooling=True, dropout=0.4):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.use_mean_pooling = use_mean_pooling
        hidden_size = self.bert.config.hidden_size
        # Head avec couche intermédiaire + dropout pour mieux généraliser
        self.corn_head = nn.Sequential(
            nn.Linear(hidden_size, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes - 1),
        )
    
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        if self.use_mean_pooling:
            # Mean pooling : moyenne des tokens (mieux pour avis mixtes)
            mask = attention_mask.unsqueeze(-1).expand(outputs.last_hidden_state.size()).float()
            sum_emb = torch.sum(outputs.last_hidden_state * mask, 1)
            sum_mask = torch.clamp(mask.sum(1), min=1e-9)
            pooled = sum_emb / sum_mask
        else:
            pooled = outputs.last_hidden_state[:, 0, :]
        return self.corn_head(pooled)

In [7]:
"""class RegressionModel(nn.Module):
    def __init__(self, model_name="answerdotai/ModernBERT-base", num_classes=5):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.head = nn.Sequential(
            nn.Linear(self.bert.config.hidden_size, 1024),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(1024, num_classes - 1)
        )
    
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        
        # Mean Pooling : On fait la moyenne de tous les tokens au lieu de prendre juste le [CLS]
        # Cela aide pour les avis mixtes (ex: échantillon 465)
        mask = attention_mask.unsqueeze(-1).expand(outputs.last_hidden_state.size()).float()
        sum_embeddings = torch.sum(outputs.last_hidden_state * mask, 1)
        sum_mask = torch.clamp(mask.sum(1), min=1e-9)
        mean_pooled = sum_embeddings / sum_mask
        
        return self.head(mean_pooled)
        """

'class RegressionModel(nn.Module):\n    def __init__(self, model_name="answerdotai/ModernBERT-base", num_classes=5):\n        super().__init__()\n        self.bert = AutoModel.from_pretrained(model_name)\n        self.head = nn.Sequential(\n            nn.Linear(self.bert.config.hidden_size, 1024),\n            nn.GELU(),\n            nn.Dropout(0.2),\n            nn.Linear(1024, num_classes - 1)\n        )\n    \n    def forward(self, input_ids, attention_mask):\n        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)\n        \n        # Mean Pooling : On fait la moyenne de tous les tokens au lieu de prendre juste le [CLS]\n        # Cela aide pour les avis mixtes (ex: échantillon 465)\n        mask = attention_mask.unsqueeze(-1).expand(outputs.last_hidden_state.size()).float()\n        sum_embeddings = torch.sum(outputs.last_hidden_state * mask, 1)\n        sum_mask = torch.clamp(mask.sum(1), min=1e-9)\n        mean_pooled = sum_embeddings / sum_mask\n       

In [24]:
# %%
import gc
# Libérer la VRAM avant de charger le modèle (important si vous ré-exécutez la cellule)
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()

def predict_rating_from_logits(logits):
    probs = torch.sigmoid(logits)
    return torch.sum(probs > 0.5, dim=1) + 1  # étoiles 1-5

device = 'cuda' if torch.cuda.is_available() else 'cpu'
num_classes = 5
model = RegressionModel(model_id, num_classes=num_classes, use_mean_pooling=True, dropout=0.4).to(device=device)
model.bert.gradient_checkpointing_enable()  # économise ~40% VRAM (plus lent)
batch_size = 4   # GPU 6 Go : batch 4 + max_length 128
epochs = 3
lr = 1.5e-5
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.02)
num_training_steps = epochs * (len(data_train) // batch_size)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * num_training_steps), num_training_steps=num_training_steps)

# CORN requires converting the rank to a set of binary labels
def label_to_corn(labels, num_classes):
    sets = []
    for i in range(num_classes - 1):
        sets.append(labels > i)
    return torch.stack(sets, dim=1).float()

all_train_loss = []
all_val_loss = []
best_val_loss = float('inf')
best_state = None
patience = 1   # arrêt dès que la val loss empire (surapprentissage très tôt)
patience_left = patience

for epoch in range(epochs):
    model.train()
    total_loss = 0
    indices = np.arange(len(data_train))
    np.random.shuffle(indices)
    
    for start in range(0, len(indices), batch_size):
        optimizer.zero_grad()
        batch_idx = indices[start:start + batch_size]
        batch_text = data_train['Review'].iloc[batch_idx]
        inputs = tokenize_data(batch_text).to(device)
        targets = train_labels[batch_idx].to(device)
        corn_targets = label_to_corn(targets, num_classes).to(device)
        logits = model(**inputs)
        loss = nn.BCEWithLogitsLoss()(logits, corn_targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    avg_train_loss = total_loss / max(1, len(indices) // batch_size)
    all_train_loss.append(avg_train_loss)
    
    # Validation + accuracy
    model.eval()
    val_loss = 0
    val_preds, val_true = [], []
    with torch.no_grad():
        for start in range(0, len(data_val), batch_size):
            batch_text = data_val['Review'].iloc[start:start+batch_size]
            inputs = tokenize_data(batch_text).to(device)
            targets = val_labels[start:start+batch_size].to(device)
            corn_targets = label_to_corn(targets, num_classes).to(device)
            logits = model(**inputs)
            loss = nn.BCEWithLogitsLoss()(logits, corn_targets)
            val_loss += loss.item()
            preds = predict_rating_from_logits(logits)
            val_preds.extend((preds).cpu().numpy())
            val_true.extend((targets + 1).cpu().numpy())  # remettre en 1-5
            
    avg_val_loss = val_loss / max(1, len(data_val) // batch_size)
    all_val_loss.append(avg_val_loss)
    val_acc = 100 * np.mean(np.array(val_preds) == np.array(val_true))
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_left = patience
    else:
        patience_left -= 1
    print(f"Epoch {epoch+1}: Train Loss {avg_train_loss:.4f} | Val Loss {avg_val_loss:.4f} | Val Acc {val_acc:.2f}%")
    if patience_left <= 0:
        print("Early stopping.")
        break

if best_state is not None:
    model.load_state_dict(best_state)
    model.to(device)

Loading weights: 100%|██████████| 134/134 [00:00<00:00, 384.21it/s, Materializing param=layers.21.mlp_norm.weight]     
ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
decoder.bias      | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1: Train Loss 0.2991 | Val Loss 0.2259 | Val Acc 66.40%
Epoch 2: Train Loss 0.2020 | Val Loss 0.2273 | Val Acc 67.50%
Early stopping.


In [32]:
# %%
def predict_rating(logits):
    probs = torch.sigmoid(logits)
    return torch.sum(probs > 0.5, dim=1) + 1

model.eval()
all_preds = []
all_true = data_test['Rating'].values

with torch.no_grad():
    for start in range(0, len(data_test), batch_size):
        batch_text = data_test['Review'].iloc[start:start+batch_size]
        inputs = tokenize_data(batch_text).to(device)
        logits = model(**inputs)
        preds = predict_rating(logits)
        all_preds.extend(preds.cpu().numpy())

all_preds = np.array(all_preds)
acc_exact = 100 * np.mean(all_preds == all_true)
acc_within_1 = 100 * np.mean(np.abs(all_preds - all_true) <= 1)
print(f"Test R2: {r2_score(all_true, all_preds):.4f} | Acc (exact): {acc_exact:.2f}% | Acc (±1 étoile): {acc_within_1:.2f}%")

Test R2: 0.6344 | Acc (exact): 63.80% | Acc (±1 étoile): 96.70%


In [33]:
from sklearn.metrics import mean_absolute_error

# Inside your test/eval loop:
mae = mean_absolute_error(all_true, all_preds)
print(f"Average Star Error (MAE): {mae:.2f}")

Average Star Error (MAE): 0.41


In [34]:
# Code pour trouver les avis qui font chuter votre score
errors = np.abs(np.array(all_true) - np.array(all_preds))
hard_cases = data_test.iloc[np.where(errors > 1.5)]
print(hard_cases['Review'].head(24))

9      palace palace wife just returned new riu palac...
10     sure read just got yesterday wait post review ...
74     n't believe reviews read terror tinytown staff...
80     dirty french press time stayed hotel 1000. pro...
120    cranes close comfort** outside window huge cra...
172    favorite 11 nights waikiki stayed 4 nights.ok ...
310    horrible file lawsuit, extremely disappointed ...
326    awesome bed great doorman dont touch mini frid...
430    pass ach leidse square pass hotel, rooms small...
440    faultless struggle criticism hotel endorse rev...
549    dont reviews, firstly goning start word warnin...
572    think twice booking breezes, just got breezes ...
582    cant wait, booked holiday n't believe reasonab...
593    great beach wish say resort just returned domi...
598    start, just got meant dream holiday upgraded r...
599    2nd time, going bk year 3rd time, start, first...
605    would'nt paid, husband friends just arrived ho...
614    breezes punta cana good 

## Modèle 2 : Correcteur
Le second modèle prend en entrée les **sorties du modèle 1** (logits/probas + prédiction) et apprend à corriger ses erreurs. Il est léger (pas de BERT) et s'entraîne vite.

In [35]:
# Extraire les sorties du modèle 1 : logits, probas, prédiction, incertitude (entropy)
def get_model1_features(model, data_df, batch_size=4):
    """Retourne X [N, 10] : 4 logits + 4 probas + pred_norm + entropy, et pred_rating [N]."""
    model.eval()
    all_logits_raw, all_probs, all_preds = [], [], []
    with torch.no_grad():
        for start in range(0, len(data_df), batch_size):
            batch_text = data_df['Review'].iloc[start:start + batch_size]
            inputs = tokenize_data(batch_text).to(device)
            logits = model(**inputs)
            probs = torch.sigmoid(logits).cpu().numpy()
            logits_np = logits.cpu().numpy()
            preds = (np.sum(probs > 0.5, axis=1) + 1).astype(np.int64)
            all_logits_raw.append(logits_np)
            all_probs.append(probs)
            all_preds.append(preds)
    L = np.vstack(all_logits_raw)   # [N, 4]
    P = np.vstack(all_probs)       # [N, 4]
    X_pred = np.concatenate(all_preds)
    # Entropie des 4 seuils CORN (incertitude) : plus c'est haut, plus le modèle hésite
    eps = 1e-9
    entropy = -np.sum(P * np.log(P + eps) + (1 - P) * np.log(1 - P + eps), axis=1, keepdims=True)
    pred_col = ((X_pred - 1) / 4.0).reshape(-1, 1)
    X = np.concatenate([L, P, pred_col, entropy / 4.0], axis=1)  # [N, 10], entropy normalisée
    return X.astype(np.float32), X_pred

X_train, pred_train = get_model1_features(model, data_train)
X_val,   pred_val   = get_model1_features(model, data_val)
X_test,  pred_test  = get_model1_features(model, data_test)
y_train = data_train['Rating'].values
y_val   = data_val['Rating'].values
y_test  = data_test['Rating'].values
print("Features correcteur - Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)

Features correcteur - Train: (18491, 10) Val: (1000, 10) Test: (1000, 10)


In [39]:
# Correcteur prédit un DÉLTA à ajouter à la prédiction du modèle 1 : -2, -1, 0, +1, +2
# Cible : delta = vrai_rating - pred_model1, borné à [-2, 2] → indice 0..4
delta_train = np.clip(y_train - pred_train, -2, 2).astype(np.int64)  # -2..2
delta_val   = np.clip(y_val - pred_val, -2, 2).astype(np.int64)
delta_train_idx = (delta_train + 2).astype(np.int64)  # 0..4 pour CrossEntropy
delta_val_idx   = (delta_val + 2).astype(np.int64)

# Poids pour forcer le correcteur à apprendre les corrections (delta != 0)
# Sinon il prédit toujours 0 car c'est la classe majoritaire
counts = np.bincount(delta_train_idx, minlength=5).astype(float) + 1
class_weights = (1.0 / counts)
class_weights = torch.tensor(class_weights / class_weights.sum() * 5, dtype=torch.float32).to(device)
print("Poids par delta (-2,-1,0,+1,+2):", class_weights.cpu().numpy())

class CorrectorModel(nn.Module):
    """Prédit le delta (-2 à +2) à ajouter à la prédiction du modèle 1."""
    def __init__(self, input_dim=10, num_deltas=5, hidden=64, dropout=0.4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, num_deltas),
        )
    def forward(self, x):
        return self.net(x)

corrector = CorrectorModel(input_dim=10, num_deltas=5, hidden=64).to(device)
opt_c = torch.optim.Adam(corrector.parameters(), lr=1e-2)
y_train_idx = torch.tensor(delta_train_idx, dtype=torch.long).to(device)
y_val_idx   = torch.tensor(delta_val_idx, dtype=torch.long).to(device)

Poids par delta (-2,-1,0,+1,+2): [2.316727   0.15778515 0.03330844 0.17545246 2.316727  ]


In [40]:
# Entraînement du correcteur avec loss pondérée (pour apprendre à corriger, pas seulement "delta=0")
from torch.utils.data import TensorDataset, DataLoader
train_ds = TensorDataset(torch.tensor(X_train), y_train_idx)
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)
loss_fn_c = nn.CrossEntropyLoss(weight=class_weights)
best_val_acc = 0
best_corrector_state = None
for epoch in range(120):
    corrector.train()
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        opt_c.zero_grad()
        loss = loss_fn_c(corrector(xb), yb)
        loss.backward()
        opt_c.step()
    corrector.eval()
    with torch.no_grad():
        delta_idx_val = corrector(torch.tensor(X_val).to(device)).argmax(dim=1).cpu().numpy()
    delta_val_pred = delta_idx_val - 2  # -2..+2
    pred_corrected_val = np.clip(pred_val + delta_val_pred, 1, 5)
    val_acc = 100 * np.mean(pred_corrected_val == y_val)
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_corrector_state = {k: v.cpu().clone() for k, v in corrector.state_dict().items()}
    if (epoch + 1) % 30 == 0:
        print(f"Epoch {epoch+1} | Val Acc (modèle1+delta): {val_acc:.2f}%")
if best_corrector_state is not None:
    corrector.load_state_dict(best_corrector_state)
    corrector.to(device)
print(f"Meilleure accuracy validation (correcteur): {best_val_acc:.2f}%")

Epoch 30 | Val Acc (modèle1+delta): 30.70%
Epoch 60 | Val Acc (modèle1+delta): 20.20%
Epoch 90 | Val Acc (modèle1+delta): 48.80%
Epoch 120 | Val Acc (modèle1+delta): 19.40%
Meilleure accuracy validation (correcteur): 51.00%


In [44]:
# Évaluation : modèle 1 seul vs modèle 1 + correcteur (delta)
# N'appliquer la correction que si le correcteur est confiant (évite la sur-correction)
CONFIDENCE_THRESHOLD = 0.55  # si max(proba) < seuil, on garde la prédiction du modèle 1
corrector.eval()
with torch.no_grad():
    logits_c = corrector(torch.tensor(X_test).to(device))
    probs_c = torch.softmax(logits_c, dim=1).cpu().numpy()
    delta_idx_test = probs_c.argmax(axis=1)
    max_prob = probs_c.max(axis=1)
delta_test = delta_idx_test - 2
# Ne corriger que quand le correcteur est confiant, sinon garder modèle 1
apply_correction = max_prob >= CONFIDENCE_THRESHOLD
pred_corrected = np.where(apply_correction, np.clip(pred_test + delta_test, 1, 5), pred_test).astype(np.int64)

acc1 = 100 * np.mean(pred_test == y_test)
acc2 = 100 * np.mean(pred_corrected == y_test)
acc1_1star = 100 * np.mean(np.abs(pred_test - y_test) <= 1)
acc2_1star = 100 * np.mean(np.abs(pred_corrected - y_test) <= 1)
r2_1 = r2_score(y_test, pred_test)
r2_2 = r2_score(y_test, pred_corrected)

print("=== Comparaison test ===")
print(f"Modèle 1 seul      : R2 = {r2_1:.4f} | Acc exacte = {acc1:.2f}% | Acc ±1 étoile = {acc1_1star:.2f}%")
print(f"Modèle 1+Correcteur: R2 = {r2_2:.4f} | Acc exacte = {acc2:.2f}% | Acc ±1 étoile = {acc2_1star:.2f}%")
print(f"Gain accuracy exacte: {acc2 - acc1:+.2f}%")
print(f"Corrections appliquées (confiance > {CONFIDENCE_THRESHOLD}): {apply_correction.sum()} / {len(pred_test)}")
if apply_correction.any():
    u, c = np.unique(delta_test[apply_correction], return_counts=True)
    print("Deltas appliqués:", dict(zip(u.tolist(), c.tolist())))
else:
    print("Deltas appliqués: aucune")

=== Comparaison test ===
Modèle 1 seul      : R2 = 0.6344 | Acc exacte = 63.80% | Acc ±1 étoile = 96.70%
Modèle 1+Correcteur: R2 = 0.6344 | Acc exacte = 63.80% | Acc ±1 étoile = 96.70%
Gain accuracy exacte: +0.00%
Corrections appliquées (confiance > 0.55): 3 / 1000
Deltas appliqués: {0: 3}


In [ ]:
import time
import random
import copy
import gc
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import r2_score

# --- CONFIGURATION DE LA BATAILLE ---
NUM_TRIALS = 15          # On réduit car BERT est lent
MAX_EPOCHS = 5           # BERT converge très vite (souvent 2-3 époques suffisent)
BATCH_SIZE = 2           # GPU 6 Go (éviter OOM)
num_classes = 5

global_best_r2 = -float('inf')
global_best_model_state = None
global_best_params = {}

print(f"⚔️ DÉBUT DE LA BATTLE ROYALE MODERNBERT")
print("="*60)

start_time = time.time()

for trial in range(1, NUM_TRIALS + 1):
    if torch.cuda.is_available():
        gc.collect()
        torch.cuda.empty_cache()
    
    # 1. PARAMÈTRES ALÉATOIRES
    seed = random.randint(0, 10000)
    # Range spécifique pour BERT (entre 1e-6 et 5e-5)
    lr = 10 ** np.random.uniform(-5.7, -4.3) 
    
    # 2. FIXER LA GRAINE
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        
    # 3. CRÉATION DU MODÈLE
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = RegressionModel(model_id, num_classes=num_classes, use_mean_pooling=True, dropout=0.2).to(device)
    model.bert.gradient_checkpointing_enable()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    loss_fn = nn.BCEWithLogitsLoss()
    
    best_val_loss_trial = float('inf')
    best_model_trial_state = None 
    
    # --- ENTRAÎNEMENT ---
    for epoch in range(MAX_EPOCHS):
        model.train()
        total_train_loss = 0
        
        # Mélange des données
        indices = np.arange(len(data_train))
        np.random.shuffle(indices)
        
        for start in range(0, len(indices), BATCH_SIZE):
            batch_idx = indices[start:start + BATCH_SIZE]
            inputs = tokenize_data(data_train['Review'].iloc[batch_idx]).to(device)
            targets = train_labels[batch_idx].to(device)
            corn_targets = label_to_corn(targets, num_classes).to(device)

            optimizer.zero_grad()
            logits = model(**inputs)
            loss = loss_fn(logits, corn_targets)
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()

        # --- VALIDATION ---
        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for start in range(0, len(data_val), BATCH_SIZE):
                inputs = tokenize_data(data_val['Review'].iloc[start:start+BATCH_SIZE]).to(device)
                targets = val_labels[start:start+BATCH_SIZE].to(device)
                corn_targets = label_to_corn(targets, num_classes).to(device)
                
                logits = model(**inputs)
                total_val_loss += loss_fn(logits, corn_targets).item()
        
        avg_val_loss = total_val_loss / (len(data_val)/BATCH_SIZE)
        
        if avg_val_loss < best_val_loss_trial:
            best_val_loss_trial = avg_val_loss
            best_model_trial_state = copy.deepcopy(model.state_dict())

    # --- TEST DU CHAMPION LOCAL ---
    model.load_state_dict(best_model_trial_state)
    model.eval()
    all_preds = []
    with torch.no_grad():
        for start in range(0, len(data_test), BATCH_SIZE):
            inputs = tokenize_data(data_test['Review'].iloc[start:start+BATCH_SIZE]).to(device)
            logits = model(**inputs)
            all_preds.extend(predict_rating(logits).cpu().numpy())
    
    current_test_r2 = r2_score(data_test['Rating'].values, all_preds)
    
    # --- VERDICT ---
    if current_test_r2 > global_best_r2:
        print(f"🔥 [Trial {trial:2d}] NOUVEAU ROI ! R2: {current_test_r2:.4f} | LR: {lr:.6f} | Seed: {seed}")
        global_best_r2 = current_test_r2
        global_best_model_state = best_model_trial_state
        global_best_params = {'seed': seed, 'lr': lr}
    else:
        print(f"   [Trial {trial:2d}] Test R2: {current_test_r2:.4f} (Best: {global_best_r2:.4f})")

    # Nettoyage mémoire pour le prochain combat
    del model
    gc.collect()
    torch.cuda.empty_cache()

duration = (time.time() - start_time) / 60
print("\n" + "="*60)
print(f"🏁 FINI en {duration:.1f} min. Meilleur R2: {global_best_r2:.4f}")
print(f"⚙️ Paramètres : {global_best_params}")

# Sauvegarde finale
torch.save(global_best_model_state, 'best_modernbert_corn.pth')

⚔️ DÉBUT DE LA BATTLE ROYALE MODERNBERT


Loading weights: 100%|██████████| 134/134 [00:00<00:00, 492.64it/s, Materializing param=layers.21.mlp_norm.weight]     
ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
head.norm.weight  | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


OutOfMemoryError: CUDA out of memory. Tried to allocate 148.00 MiB. GPU 0 has a total capacity of 5.64 GiB of which 124.44 MiB is free. Including non-PyTorch memory, this process has 5.50 GiB memory in use. Of the allocated memory 5.17 GiB is allocated by PyTorch, and 224.21 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

- 🔥 [Trial  1] NOUVEAU ROI ! R2: 0.7046 | LR: 0.000003 | Seed: 2817
- 🔥 [Trial  3] NOUVEAU ROI ! R2: 0.7053 | LR: 0.000018 | Seed: 422
- 🔥 [Trial  1] NOUVEAU ROI ! R2: 0.7294 | LR: 0.000009 | Seed: 7483
- 🔥 [Trial  3] NOUVEAU ROI ! R2: 0.7357 | LR: 0.000038 | Seed: 2726




In [ ]:
#torch.save(model.state_dict(),'model_v1/model_v5.pth')

In [ ]:
print(f"Mémoire allouée: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
print(f"Mémoire réservée: {torch.cuda.memory_reserved() / 1024**2:.2f} MB")


Mémoire allouée: 2407.16 MB
Mémoire réservée: 2774.00 MB
